# Experiment 13: DeBERTa-v3 Architecture Comparison

Re-run the clean held-out evaluation (same protocol as NB09c) with an additional architecture:
**DeBERTa-v3-base** alongside BERT-base and ModernBERT-base.

All three models use LoRA r=16 with identical hyperparameters, trained on the aggregated
dataset (excluding FPB source 5) and evaluated on FPB `sentences_50agree` and `sentences_allAgree`.

This adds DeBERTa-v3-base to the BERT vs ModernBERT comparison under identical conditions,
providing a stronger baseline from the DeBERTa family which uses disentangled attention
and enhanced mask decoder.

In [ ]:
%%capture
!pip install -q "datasets>=3.4.1,<4.0.0" scikit-learn matplotlib seaborn peft accelerate transformers sentencepiece protobuf

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, training_args
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import json
import gc
import matplotlib.pyplot as plt
import seaborn as sns

NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
FPB_SOURCE = 5

## 1. Data Loading

In [ ]:
label_dict = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}

ds = load_dataset("neoyipeng/financial_reasoning_aggregated")
ds = ds.filter(lambda x: x["task"] == "sentiment")
ds = ds.filter(lambda x: x["source"] != FPB_SOURCE)

remove_cols = [c for c in ds["train"].column_names if c not in ("text", "labels")]
ds = ds.map(
    lambda ex: {
        "text": ex["text"],
        "labels": np.eye(NUM_CLASSES)[label_dict[ex["label"]]],
    },
    remove_columns=remove_cols,
)

print(f"Train: {len(ds['train']):,}  |  Val: {len(ds['validation']):,}  |  Test: {len(ds['test']):,}")

fpb_50 = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
fpb_all = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]
print(f"FPB 50agree: {len(fpb_50):,}  |  FPB allAgree: {len(fpb_all):,}")

## 2. Evaluation Helper

In [ ]:
def evaluate_on_fpb(model, tokenizer, fpb_dataset, batch_size=32):
    fpb_texts = fpb_dataset["sentence"]
    fpb_labels = fpb_dataset["label"]
    all_preds = []

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(fpb_texts), batch_size), desc="Evaluating"):
            batch = fpb_texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs = {k: v.cuda() for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)

    y_true = np.array(fpb_labels)
    y_pred = np.array(all_preds)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    report = classification_report(y_true, y_pred, target_names=LABEL_NAMES)
    cm = confusion_matrix(y_true, y_pred)

    print(f"Accuracy: {acc:.4f} ({int(acc * len(y_true))}/{len(y_true)})")
    print(f"Macro F1: {macro_f1:.4f}")
    print(report)

    return {"accuracy": acc, "macro_f1": macro_f1, "cm": cm, "y_true": y_true, "y_pred": y_pred}

## 3. Train & Evaluate BERT-base + LoRA

In [ ]:
print("Training BERT-base-uncased + LoRA r=16...")
bert_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=NUM_CLASSES)
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

bert_lora = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["query", "key", "value", "dense"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.SEQ_CLS,
)
bert_model = get_peft_model(bert_model, bert_lora)
bert_model.gradient_checkpointing_enable()
bert_model = bert_model.cuda()
bert_model.print_trainable_parameters()

def tokenize_bert(examples):
    return bert_tokenizer(examples["text"], truncation=True, max_length=512)

train_data = ds["train"].map(tokenize_bert, batched=True)
val_data = ds["validation"].map(tokenize_bert, batched=True)

bert_trainer = Trainer(
    model=bert_model,
    processing_class=bert_tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=TrainingArguments(
        output_dir="trainer_output_13_bert",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        fp16=True,
        optim=training_args.OptimizerNames.ADAMW_TORCH,
        learning_rate=2e-4,
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=3407,
        num_train_epochs=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="epoch",
        gradient_checkpointing=True,
        report_to="none",
        save_total_limit=2,
    ),
    compute_metrics=lambda eval_pred: {
        "accuracy": accuracy_score(
            eval_pred[1].argmax(axis=-1), eval_pred[0].argmax(axis=-1)
        )
    },
)

bert_trainer.train()
bert_model = bert_model.cuda().eval()

In [ ]:
print("\n" + "=" * 60)
print("BERT-base + LoRA -- FPB sentences_50agree")
print("=" * 60)
bert_50 = evaluate_on_fpb(bert_model, bert_tokenizer, fpb_50)

print("\n" + "=" * 60)
print("BERT-base + LoRA -- FPB sentences_allAgree")
print("=" * 60)
bert_all = evaluate_on_fpb(bert_model, bert_tokenizer, fpb_all)

del bert_model, bert_trainer
gc.collect()
torch.cuda.empty_cache()

## 4. Train & Evaluate ModernBERT-base + LoRA

In [ ]:
print("Training ModernBERT-base + LoRA r=16...")
mb_model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base", num_labels=NUM_CLASSES)
mb_tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

mb_lora = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.SEQ_CLS,
)
mb_model = get_peft_model(mb_model, mb_lora)
mb_model.gradient_checkpointing_enable()
mb_model = mb_model.cuda()
mb_model.print_trainable_parameters()

def tokenize_mb(examples):
    return mb_tokenizer(examples["text"], truncation=True, max_length=512)

train_data_mb = ds["train"].map(tokenize_mb, batched=True)
val_data_mb = ds["validation"].map(tokenize_mb, batched=True)

mb_trainer = Trainer(
    model=mb_model,
    processing_class=mb_tokenizer,
    train_dataset=train_data_mb,
    eval_dataset=val_data_mb,
    args=TrainingArguments(
        output_dir="trainer_output_13_modernbert",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        fp16=True,
        optim=training_args.OptimizerNames.ADAMW_TORCH,
        learning_rate=2e-4,
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=3407,
        num_train_epochs=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="epoch",
        gradient_checkpointing=True,
        report_to="none",
        save_total_limit=2,
    ),
    compute_metrics=lambda eval_pred: {
        "accuracy": accuracy_score(
            eval_pred[1].argmax(axis=-1), eval_pred[0].argmax(axis=-1)
        )
    },
)

mb_trainer.train()
mb_model = mb_model.cuda().eval()

In [ ]:
print("\n" + "=" * 60)
print("ModernBERT-base + LoRA -- FPB sentences_50agree")
print("=" * 60)
mb_50 = evaluate_on_fpb(mb_model, mb_tokenizer, fpb_50)

print("\n" + "=" * 60)
print("ModernBERT-base + LoRA -- FPB sentences_allAgree")
print("=" * 60)
mb_all = evaluate_on_fpb(mb_model, mb_tokenizer, fpb_all)

del mb_model, mb_trainer
gc.collect()
torch.cuda.empty_cache()

## 5. Train & Evaluate DeBERTa-v3-base + LoRA

In [ ]:
print("Training DeBERTa-v3-base + LoRA r=16...")
# NOTE: DeBERTa-v3 requires special handling:
# 1. fp16=False (disentangled attention is numerically unstable in FP16)
# 2. No gradient checkpointing (known compatibility issue with DeBERTa-v3 + PEFT)
# 3. Reduced batch size to compensate for no grad checkpointing + no FP16
db_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base", num_labels=NUM_CLASSES,
)
db_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

db_lora = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["query_proj", "key_proj", "value_proj", "dense"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.SEQ_CLS,
)
db_model = get_peft_model(db_model, db_lora)
# Do NOT enable gradient_checkpointing for DeBERTa-v3
db_model = db_model.cuda()
db_model.print_trainable_parameters()

def tokenize_db(examples):
    return db_tokenizer(examples["text"], truncation=True, max_length=512)

train_data_db = ds["train"].map(tokenize_db, batched=True)
val_data_db = ds["validation"].map(tokenize_db, batched=True)

db_trainer = Trainer(
    model=db_model,
    processing_class=db_tokenizer,
    train_dataset=train_data_db,
    eval_dataset=val_data_db,
    args=TrainingArguments(
        output_dir="trainer_output_13_deberta",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        fp16=False,
        optim=training_args.OptimizerNames.ADAMW_TORCH,
        learning_rate=2e-4,
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=3407,
        num_train_epochs=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="epoch",
        gradient_checkpointing=False,
        report_to="none",
        save_total_limit=1,
    ),
    compute_metrics=lambda eval_pred: {
        "accuracy": accuracy_score(
            eval_pred[1].argmax(axis=-1), eval_pred[0].argmax(axis=-1)
        )
    },
)

db_trainer.train()
db_model = db_model.cuda().eval()

In [ ]:
print("\n" + "=" * 60)
print("DeBERTa-v3-base + LoRA -- FPB sentences_50agree")
print("=" * 60)
db_50 = evaluate_on_fpb(db_model, db_tokenizer, fpb_50)

print("\n" + "=" * 60)
print("DeBERTa-v3-base + LoRA -- FPB sentences_allAgree")
print("=" * 60)
db_all = evaluate_on_fpb(db_model, db_tokenizer, fpb_all)

del db_model, db_trainer
gc.collect()
torch.cuda.empty_cache()

## 6. Comparison Table

In [ ]:
print("=" * 70)
print("EXPERIMENT 13: ARCHITECTURE COMPARISON (BERT vs ModernBERT vs DeBERTa)")
print("=" * 70)
print(f"{'Model':<30} {'FPB 50agree':>12} {'FPB allAgree':>12}")
print("-" * 60)
print(f"{'BERT-base + LoRA r=16':<30} {bert_50['accuracy']:>11.2%} {bert_all['accuracy']:>12.2%}")
print(f"{'ModernBERT + LoRA r=16':<30} {mb_50['accuracy']:>11.2%} {mb_all['accuracy']:>12.2%}")
print(f"{'DeBERTa-v3 + LoRA r=16':<30} {db_50['accuracy']:>11.2%} {db_all['accuracy']:>12.2%}")
print("-" * 60)
print(f"{'Gap (BERT - MB)':<30} {(bert_50['accuracy']-mb_50['accuracy'])*100:>10.2f}pp {(bert_all['accuracy']-mb_all['accuracy'])*100:>10.2f}pp")
print(f"{'Gap (BERT - DB)':<30} {(bert_50['accuracy']-db_50['accuracy'])*100:>10.2f}pp {(bert_all['accuracy']-db_all['accuracy'])*100:>10.2f}pp")
print(f"{'Gap (MB - DB)':<30} {(mb_50['accuracy']-db_50['accuracy'])*100:>10.2f}pp {(mb_all['accuracy']-db_all['accuracy'])*100:>10.2f}pp")

results_13 = {
    "bert_50agree": bert_50["accuracy"],
    "bert_allagree": bert_all["accuracy"],
    "bert_50_f1": bert_50["macro_f1"],
    "bert_all_f1": bert_all["macro_f1"],
    "mb_50agree": mb_50["accuracy"],
    "mb_allagree": mb_all["accuracy"],
    "mb_50_f1": mb_50["macro_f1"],
    "mb_all_f1": mb_all["macro_f1"],
    "db_50agree": db_50["accuracy"],
    "db_allagree": db_all["accuracy"],
    "db_50_f1": db_50["macro_f1"],
    "db_all_f1": db_all["macro_f1"],
}
with open("results_13_deberta.json", "w") as f:
    json.dump(results_13, f, indent=2)
print("\nSaved to results_13_deberta.json")

## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (res, title) in zip(axes, [
    (bert_50, "BERT-base + LoRA r=16"),
    (mb_50, "ModernBERT + LoRA r=16"),
    (db_50, "DeBERTa-v3 + LoRA r=16"),
]):
    sns.heatmap(
        res["cm"], annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax,
    )
    ax.set_title(f"{title}\nAcc={res['accuracy']:.2%}  F1={res['macro_f1']:.2%}")
    ax.set_ylabel("True")
    ax.set_xlabel("Predicted")

plt.suptitle("Exp 13: Architecture Comparison -- FPB sentences_50agree", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("13_deberta_baseline.png", dpi=150, bbox_inches="tight")
plt.show()